In [ ]:
# ==============================================================================
# COLAB / ANTIGRAVITY TRAINING — LAUNCHER
#
# This notebook contains NO training code. It runs the script from the
# repository: scripts/colab/colab_clean_cell.py
#
# Why: this notebook used to hold a pasted copy, and the copy went stale --
# on 2026-08-02 it still used train_test_split(random_state=42), a random
# split that lets a validation window share 19 of its 20 days with a
# training window, months after the script had been fixed to split
# chronologically with a purge gap. One copy, in git, reviewed and tested.
#
# WHERE THE PROJECT IS: found, not assumed. Earlier versions of this cell
# required the repository on Google Drive and stopped otherwise; that is one
# valid setup among several. An IDE-hosted notebook usually has the checkout
# as the working directory and only the batch data on Drive.
# ==============================================================================

import os, sys
from pathlib import Path

# Set this if neither guess below finds your checkout.
PROJECT_PATH = None

def _looks_like_project(p):
    p = Path(p)
    return (p / "scripts" / "colab" / "colab_clean_cell.py").exists()

def _find_project():
    if PROJECT_PATH and _looks_like_project(PROJECT_PATH):
        return Path(PROJECT_PATH)
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if _looks_like_project(candidate):
            return candidate
    drive = Path("/content/drive/MyDrive/trading_project")
    if _looks_like_project(drive):
        return drive
    try:
        from google.colab import drive as _d
        _d.mount("/content/drive", force_remount=False)
        if _looks_like_project(drive):
            return drive
    except Exception:
        pass
    return None

project = _find_project()
if project is None:
    raise SystemExit(
        "Could not find the repository. Looked in the working directory and "
        "its parents, then /content/drive/MyDrive/trading_project. Set "
        "PROJECT_PATH at the top of this cell to your checkout."
    )

os.chdir(project)
for path in (str(project), str(project / "src")):
    if path not in sys.path:
        sys.path.insert(0, path)

# One log holding everything the run says. The trainer reports progress with
# print(), which otherwise goes only to this cell's output and is lost the
# moment it scrolls or the session ends -- and a two-hour run that ends in
# "0 predictions" is exactly the one whose log is wanted afterwards.
sys.path.insert(0, str(project / "scripts" / "colab"))
from run_logging import start_run_log
LOG_PATH = start_run_log(name="colab_train")

import subprocess, datetime
print("project:", project)
print("script mtime:", datetime.datetime.fromtimestamp(
    (project / "scripts" / "colab" / "colab_clean_cell.py").stat().st_mtime))
try:
    head = subprocess.run(
        ["git", "-C", str(project), "log", "-1", "--format=%h %ad %s", "--date=short"],
        capture_output=True, text=True, timeout=30)
    print("repo HEAD:", (head.stdout or head.stderr).strip() or "(not a git checkout)")
except Exception as exc:
    print("repo HEAD: unavailable --", exc)


In [ ]:
%run "scripts/colab/colab_clean_cell.py"


In [ ]:
# Close the log and show where it is, plus what the run produced.
from run_logging import stop_run_log
stop_run_log()
print("LOG:", LOG_PATH.resolve())

import os
batch_dir = "data/colab/accumulated/main_database"
if os.path.isdir(batch_dir):
    for f in sorted(os.listdir(batch_dir)):
        if not f.startswith('.'):
            size_mb = os.path.getsize(os.path.join(batch_dir, f)) / 1024 / 1024
            print(f"{size_mb:>8.1f} MB  {f}")
else:
    print(f"(no batch directory at {batch_dir})")
